# Notebook 01 — MTJ Fundamentals and Resistance-Variation PUF

**Graduate research baseline | Google Colab / Jupyter | Python 3**

## Objectives
1. Derive a simple magnetic tunnel junction (MTJ) resistance model.
2. Generate a population of devices with process variation.
3. Construct pair-comparison challenge-response strings.
4. Evaluate uniformity, inter-chip uniqueness, intra-chip reliability and bit aliasing.
5. Understand the model's limitations before proceeding to LLG and STT/SOT dynamics.

**Scope:** This is a statistical/circuit-level baseline inspired by resistance-comparison MTJ PUFs. Parameters are illustrative, not calibrated to a fabricated technology. It does not yet solve magnetization dynamics or model STT/SOT switching; those are later notebooks.

## 1. MTJ physics and resistance equations

An MTJ contains two ferromagnetic layers separated by a thin insulating tunnel barrier. One layer is the reference layer; the other is the free layer. Parallel (P) alignment has resistance \(R_P\); antiparallel (AP) alignment has resistance \(R_{AP}\), normally larger.

The tunnelling magnetoresistance ratio is

\[
\mathrm{TMR}=\frac{R_{AP}-R_P}{R_P},\qquad R_{AP}=R_P(1+\mathrm{TMR}).
\]

A convenient first-order angular interpolation is

\[
R(\theta)=R_P\left[1+\frac{\mathrm{TMR}}{2}(1-\cos\theta)\right],
\]

where \(\theta\) is the angle between the two magnetizations. It gives the correct P and AP limits. This is a phenomenological circuit model, not a full quantum-transport calculation.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20261110
rng = np.random.default_rng(SEED)
print("Python", sys.version.split()[0], "NumPy", np.__version__)

## 2. Implement and verify the angular resistance relation

In [ ]:
def mtj_resistance(theta_rad, rp_ohm, tmr):
    """First-order angular MTJ resistance [ohm]. TMR is a ratio, not percent."""
    return rp_ohm * (1.0 + 0.5 * tmr * (1.0 - np.cos(theta_rad)))

RP0, TMR0 = 10_000.0, 1.0  # illustrative: 10 kOhm, 100% TMR
theta = np.linspace(0, np.pi, 301)
R = mtj_resistance(theta, RP0, TMR0)
assert np.isclose(mtj_resistance(0, RP0, TMR0), RP0)
assert np.isclose(mtj_resistance(np.pi, RP0, TMR0), RP0*(1+TMR0))
print(f"R_P={mtj_resistance(0,RP0,TMR0):.1f} ohm; R_AP={mtj_resistance(np.pi,RP0,TMR0):.1f} ohm")
plt.figure(figsize=(7,4)); plt.plot(theta/np.pi,R/1e3)
plt.xlabel(r"Relative angle $\theta/\pi$"); plt.ylabel("Resistance (kOhm)")
plt.title("Angular MTJ resistance"); plt.grid(); plt.show()

## 3. Device-to-device process variation

For a first model, assume
\[
R_{P,i}\sim\mathcal{N}(\bar R_P,\sigma_{R_P}^2),\qquad
\mathrm{TMR}_i\sim\mathcal{N}(\overline{\mathrm{TMR}},\sigma_{\mathrm{TMR}}^2).
\]

Independent Gaussian variation is only a starting assumption. Real process variation may be correlated, spatially dependent, and non-Gaussian.

In [ ]:
N_CHIPS, N_DEVICES = 64, 128
RP_MEAN, RP_REL_SIGMA = 10_000.0, 0.04
TMR_MEAN, TMR_SIGMA = 1.0, 0.06

# Each chip has its own array of nominally identical MTJs.
rp = rng.normal(RP_MEAN, RP_MEAN*RP_REL_SIGMA, (N_CHIPS,N_DEVICES))
rp = np.maximum(rp, 100.0)
tmr = np.maximum(rng.normal(TMR_MEAN,TMR_SIGMA,(N_CHIPS,N_DEVICES)),0.01)
rap = rp*(1+tmr)

# Inspect chip 0's device population.
devices = pd.DataFrame({"device":np.arange(N_DEVICES),"RP_ohm":rp[0],"TMR_ratio":tmr[0],"RAP_ohm":rap[0]})
display(devices.head(10)); display(devices.describe())
fig,ax=plt.subplots(1,3,figsize=(13,3.5))
for a,x,title,xlabel in zip(ax,[rp[0]/1e3,tmr[0],rap[0]/1e3],["Parallel resistance","TMR","Antiparallel resistance"],["RP (kOhm)","TMR ratio","RAP (kOhm)"]):
    a.hist(x,bins=18); a.set_title(title); a.set_xlabel(xlabel); a.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 4. Challenge-response construction

A PUF is represented abstractly by \(R=F(C,\boldsymbol\xi)\), where \(C\) is the challenge and \(\boldsymbol\xi\) denotes the physical device parameters.

Here a challenge selects two MTJs, \((i,j)\), and returns
\[
r(C)=\begin{cases}1,&R_i>R_j,\\0,&R_i\le R_j.\end{cases}
\]

We use \(R_P\) for the baseline. Each chip has its own process-varied array; all chips receive the same challenge set.

In [ ]:
N_CHALLENGES = 512
pairs = np.array([rng.choice(N_DEVICES,2,replace=False) for _ in range(N_CHALLENGES)])

def make_response(resistances, challenge_pairs):
    left = resistances[challenge_pairs[:,0]]
    right = resistances[challenge_pairs[:,1]]
    return (left > right).astype(np.uint8)

responses = np.array([make_response(rp[k],pairs) for k in range(N_CHIPS)])
print("Response matrix:",responses.shape)
print("First 60 bits:","".join(map(str,responses[0,:60])))

## 5. PUF metrics

For binary strings \(a,b\) of length \(L\), normalized Hamming distance is
\[
HD(a,b)=\frac{1}{L}\sum_{k=1}^L(a_k\oplus b_k).
\]

- **Inter-chip HD:** same challenges, different chips; an ideal unbiased population has mean near 0.5.
- **Intra-chip HD:** repeated reads of the same chip; an ideal stable PUF has mean near zero.
- **Uniformity:** fraction of response bits equal to one.
- **Bit aliasing:** for each challenge position, fraction of chips returning one.

These metrics do not, by themselves, establish resistance to modelling attacks.

In [ ]:
def hamming(a,b):
    if np.shape(a)!=np.shape(b): raise ValueError("Vectors must have equal shape")
    return np.mean(np.asarray(a)!=np.asarray(b))

inter_hd=[hamming(responses[i],responses[j]) for i in range(N_CHIPS) for j in range(i+1,N_CHIPS)]
uniformity=responses.mean(axis=1)
bit_aliasing=responses.mean(axis=0)

# Repeat-read model: additive read noise on each cell resistance.
READ_NOISE_SIGMA=20.0 # ohm, illustrative
N_REPEATS=20
repeat_hd=[]
for chip in range(N_CHIPS):
    baseline=responses[chip]
    for _ in range(N_REPEATS):
        noisy=rp[chip]+rng.normal(0,READ_NOISE_SIGMA,N_DEVICES)
        repeat_hd.append(hamming(baseline,make_response(noisy,pairs)))

print(f"Mean inter-chip HD: {np.mean(inter_hd):.4f} (ideal target ~0.5)")
print(f"Mean intra-chip HD: {np.mean(repeat_hd):.6f} (ideal target ~0)")
print(f"Mean uniformity:    {np.mean(uniformity):.4f} (ideal target ~0.5)")
print(f"Mean bit aliasing:  {np.mean(bit_aliasing):.4f} (ideal target ~0.5)")
fig,ax=plt.subplots(1,3,figsize=(13,3.5))
ax[0].hist(inter_hd,bins=25); ax[0].axvline(.5,ls='--'); ax[0].set_title('Inter-chip HD')
ax[1].hist(repeat_hd,bins=25); ax[1].set_title('Intra-chip HD')
ax[2].hist(bit_aliasing,bins=20,range=(0,1)); ax[2].axvline(.5,ls='--'); ax[2].set_title('Bit aliasing')
for a in ax: a.set_xlabel('Normalized HD / fraction'); a.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 6. Reliability versus read noise

A comparator is vulnerable when the two compared resistances are close. Increase read-noise standard deviation and measure response bit flips.

In [ ]:
noise_sweep=np.linspace(0,250,16)
mean_flip=[]
for sigma in noise_sweep:
    flips=[]
    for chip in range(N_CHIPS):
        for _ in range(12):
            noisy=rp[chip]+rng.normal(0,sigma,N_DEVICES)
            flips.append(hamming(responses[chip],make_response(noisy,pairs)))
    mean_flip.append(np.mean(flips))
plt.figure(figsize=(7,4)); plt.plot(noise_sweep,mean_flip,marker='o')
plt.xlabel('Read-noise standard deviation (ohm)'); plt.ylabel('Mean response bit-flip fraction')
plt.title('Illustrative reliability degradation'); plt.grid(); plt.show()

## 7. Global and local process variation

A shared chip-level offset and local device mismatch are not equivalent. A uniform shift applied to every MTJ on a chip often preserves pairwise ordering, whereas local mismatch changes the ordering. This experiment separates the two effects.

In [ ]:
GLOBAL_SIGMA, LOCAL_SIGMA = 300.0, 400.0 # ohm
global_offset=rng.normal(0,GLOBAL_SIGMA,(N_CHIPS,1))
local_offset=rng.normal(0,LOCAL_SIGMA,(N_CHIPS,N_DEVICES))
rp_global_local=np.maximum(RP_MEAN+global_offset+local_offset,100)
resp_gl=np.array([make_response(rp_global_local[k],pairs) for k in range(N_CHIPS)])
hd_gl=[hamming(resp_gl[i],resp_gl[j]) for i in range(N_CHIPS) for j in range(i+1,N_CHIPS)]
print("Local-only variation mean inter-chip HD:",np.mean(inter_hd))
print("Global + local variation mean inter-chip HD:",np.mean(hd_gl))

## 8. Interpretation, limitations and validation

### What this notebook demonstrates
- The angular relation reproduces the parallel and antiparallel resistance limits.
- Device-to-device variation can generate distinct comparator response strings.
- Read noise can cause response instability, especially for pairs with small resistance margins.
- Global and local process variations have different effects.

### What it does not demonstrate
- No LLG magnetization dynamics, STT/SOT switching, domain-wall motion, or quantum transport.
- No transistor-level sense amplifier, write circuit, parasitic network, or measured process distribution.
- No proof of cryptographic security or resistance to machine-learning attacks.

### Validation checklist
1. Confirm \(R(0)=R_P\) and \(R(\pi)=R_P(1+\mathrm{TMR})\).
2. Set all devices identical and verify that comparator responses collapse to ties / a deterministic tie policy.
3. Increase read noise and verify that response instability generally rises.
4. Change the random seed and report variation across independent runs.
5. Report parameter assumptions and units with every result.

## 9. Exercises
1. Replace Gaussian \(R_P\) variation with a lognormal distribution.
2. Introduce correlation between \(R_P\) and TMR.
3. Compare random-pair challenges with adjacent-cell challenges.
4. Reject pairs with \(|R_i-R_j|<\Delta R_{min}\). Quantify the trade-off between usable challenge count and reliability.
5. Explain why empirical Shannon entropy is not automatically a conditional min-entropy security guarantee.

## 10. Foundational references
- J. C. Slonczewski, “Current-driven excitation of magnetic multilayers,” *Journal of Magnetism and Magnetic Materials* **159**, L1–L7 (1996).
- L. Berger, “Emission of spin waves by a magnetic multilayer traversed by a current,” *Physical Review B* **54**, 9353 (1996).
- J. C. Slonczewski, “Currents and torques in metallic magnetic multilayers,” *Journal of Magnetism and Magnetic Materials* **247**, 324–338 (2002).
- D. C. Ralph and M. D. Stiles, “Spin transfer torques,” *Journal of Magnetism and Magnetic Materials* **320**, 1190–1216 (2008).

Verify publisher metadata/DOIs before using these in a submitted manuscript.

## 11. Export data for subsequent notebooks

The files below preserve the illustrative device population, challenge list, response matrix and summary metrics. They are simulation outputs, not experimental data.

In [ ]:
from pathlib import Path
outdir=Path('notebook01_outputs'); outdir.mkdir(exist_ok=True)
# Export chip 0 device parameters and all chip arrays separately.
pd.DataFrame(rp,columns=[f'MTJ_{i}' for i in range(N_DEVICES)]).to_csv(outdir/'parallel_resistance_by_chip_ohm.csv',index=False)
pd.DataFrame(tmr,columns=[f'MTJ_{i}' for i in range(N_DEVICES)]).to_csv(outdir/'tmr_by_chip.csv',index=False)
pd.DataFrame(pairs,columns=['device_i','device_j']).to_csv(outdir/'challenge_pairs.csv',index=False)
pd.DataFrame(responses).to_csv(outdir/'response_matrix.csv',index=False)
pd.DataFrame([{'seed':SEED,'n_chips':N_CHIPS,'n_devices':N_DEVICES,'n_challenges':N_CHALLENGES,'mean_inter_chip_hd':np.mean(inter_hd),'mean_intra_chip_hd':np.mean(repeat_hd),'mean_uniformity':np.mean(uniformity),'mean_bit_aliasing':np.mean(bit_aliasing)}]).to_csv(outdir/'metrics_summary.csv',index=False)
print('Saved:',*[p.name for p in sorted(outdir.iterdir())],sep='\n - ')

---
### Completion checkpoint
You should be able to explain the MTJ P/AP resistance states, the meaning of TMR, how pair challenges produce bits, why inter-chip HD near 0.5 is desirable, and why low intra-chip HD matters.

**Next:** Notebook 02 — Deterministic and stochastic Landau–Lifshitz–Gilbert dynamics from first principles.